In [326]:
import pandas as pd
import numpy as np
import importlib

import plotly.graph_objects as go
import plotly.express as px
from plotly.colors import qualitative

from Irina import utility_functions as uf

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform


In [376]:
importlib.reload(uf)

<module 'Irina.utility_functions' from '/Users/irinavorobeva/PycharmProjects/geoTopics/Irina/utility_functions.py'>

In [617]:
year = 2000

df_country_subfield = pd.read_csv(uf.PATH+"df_country_subfield_Floriana/country_subfield_matrix_"+str(year)+".csv", index_col=0)
df_country_subfield_unique = pd.read_csv(uf.PATH+"df_country_subfield_Floriana/country_subfield_matrix_SingleCountry_"+str(year)+".csv", index_col=0)

In [618]:
df_country_subfield_norm = uf.get_country_subfield_prob_metric(df_country_subfield)
df_country_subfield_norm_unique = uf.get_country_subfield_prob_metric(df_country_subfield_unique)

# Compare country-subfield with and without collab

In [619]:
df_country_subfield

,1100,1102,1103,1104,1105,1106,1107,1108,1109,1110,...,3603,3604,3605,3607,3608,3609,3611,3612,3614,3616
country,,,,,,,,,,,,,,,,,,,,,
AE,0,3,1,0,0,1,0,0,1,1,...,0,0,0,0,0,0,1,0,0,1
AF,0,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
AG,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AL,0,0,0,0,0,2,0,0,1,3,...,0,0,0,0,0,0,0,0,0,0
AM,0,0,0,1,0,1,0,0,0,2,...,0,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
XK,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
YE,0,0,0,0,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ZA,10,25,28,9,84,19,5,1,19,87,...,1,5,3,0,1,5,5,0,3,8


In [622]:
df_country_stats = (
    df_country_subfield_norm
    .assign(
        entropy=lambda df: -df.mul(np.log(df.replace(0, np.nan))).sum(axis=1),
        norm_entropy=lambda df: df.entropy / np.log(len(df.columns) - 1),
        total_articles=df_country_subfield.sum(axis=1),
        log_total_articles=lambda df: np.log(df.total_articles),
        gini=df_country_subfield.apply(uf.gini, axis=1)
    )
    .merge(uf.df_country[["alpha-2", "region"]], left_index=True, right_on="alpha-2", how="left")
    .rename(columns={"alpha-2": "country"})
    .set_index("country")
    .sort_values('total_articles', ascending=False)
    [["region", "entropy", "norm_entropy", "total_articles", "log_total_articles", "gini"]]
)

df_country_stats_unique = (
    df_country_subfield_norm_unique
    .assign(
        entropy=lambda df: -df.mul(np.log(df.replace(0, np.nan))).sum(axis=1),
        norm_entropy=lambda df: df.entropy / np.log(len(df.columns) - 1),
        total_articles=df_country_subfield_unique.sum(axis=1),
        log_total_articles=lambda df: np.log(df.total_articles),
        gini=df_country_subfield_unique.apply(uf.gini, axis=1)
    )
    .merge(uf.df_country[["alpha-2", "region"]], left_index=True, right_on="alpha-2", how="left")
    .rename(columns={"alpha-2": "country"})
    .set_index("country")
    .sort_values('total_articles', ascending=False)
    [["region", "entropy", "norm_entropy", "total_articles", "log_total_articles", "gini"]]
)

In [623]:
col_list_coupled = [[col, col+"_unique"] for col in df_country_stats.columns.to_list()]
col_list = [x for sublist in col_list_coupled for x in sublist]

df_country_stats_compare = (
    df_country_stats
    .merge(df_country_stats_unique, left_index=True, right_index=True, how="outer", suffixes=("", "_unique"))
    [col_list]
    .drop("region_unique", axis=1)
)

In [624]:
(
    df_country_stats_compare
    .reset_index()
    .set_index(["country", "region"])
    .dropna(subset=["total_articles_unique"])
    .assign(
        diff_entropy=lambda df: np.abs(df.norm_entropy - df.norm_entropy_unique),
        diff_gini=lambda df: np.abs(df.gini - df.gini_unique),
        loss_articles=lambda df: (df.total_articles - df.total_articles_unique) / df.total_articles,
    )
    .sort_values("loss_articles", ascending=False)
    .query("total_articles > 1000")
    # .median(axis=0)
).head(10)

,,entropy,entropy_unique,norm_entropy,norm_entropy_unique,total_articles,total_articles_unique,log_total_articles,log_total_articles_unique,gini,gini_unique,diff_entropy,diff_gini,loss_articles
country,region,,,,,,,,,,,,,
MA,Africa,4.383071,4.321836,0.793251,0.782169,1128,506.0,7.028201,6.226537,0.746185,0.759413,0.011082,0.013227,0.551418
HK,Asia,4.495689,4.471903,0.813633,0.809328,3735,1918.0,8.225503,7.559038,0.720856,0.725760,0.004305,0.004904,0.486479
SK,Europe,4.420439,4.394990,0.800014,0.795408,1639,927.0,7.401842,6.831954,0.743253,0.745282,0.004606,0.002029,0.434411
RO,Europe,4.135215,4.113542,0.748394,0.744471,1626,981.0,7.393878,6.888572,0.807907,0.810873,0.003922,0.002965,0.396679
CZ,Europe,4.543477,4.578976,0.822281,0.828706,4449,2710.0,8.400435,7.904704,0.710003,0.699434,0.006425,0.010569,0.390874
BG,Europe,4.414211,4.399561,0.798887,0.796235,1733,1059.0,7.457609,6.965080,0.740319,0.741389,0.002651,0.001071,0.388921
HU,Europe,4.604893,4.657453,0.833397,0.842909,3828,2352.0,8.250098,7.763021,0.692499,0.674835,0.009512,0.017664,0.385580
TH,Asia,4.617358,4.555082,0.835652,0.824382,1287,801.0,7.160069,6.685861,0.685649,0.704906,0.011271,0.019257,0.377622
CH,Europe,4.634683,4.677482,0.838788,0.846534,12719,7950.0,9.450852,8.980927,0.676285,0.663736,0.007746,0.012549,0.374951


In [625]:
uf.get_country_info("HK")

,name,region,sub-region,code
100,Hong Kong,Asia,Eastern Asia,HK


In [626]:
(
    df_country_stats_compare
    .query("total_articles_unique.isna()")
    .index
    .to_frame()
    .merge(uf.df_country[["alpha-2", "name", "region", "sub-region"]], left_index=True, right_on="alpha-2", how="left")
)

,country,alpha-2,name,region,sub-region
25,BT,BT,Bhutan,Asia,Southern Asia
61,DJ,DJ,Djibouti,Africa,Sub-Saharan Africa
62,DM,DM,Dominica,Americas,Latin America and the Caribbean
118,KP,KP,"Korea, Democratic People's Republic of",Asia,Eastern Asia
135,MV,MV,Maldives,Asia,Southern Asia
229,TC,TC,Turks and Caicos Islands,Americas,Latin America and the Caribbean
191,VC,VC,Saint Vincent and the Grenadines,Americas,Latin America and the Caribbean


In [627]:
df_country_subfield_diff = (
    df_country_subfield.fillna(0)
    .sub(df_country_subfield_unique.fillna(0))
    .div(df_country_subfield)
) * 100

In [628]:
country = "AZ"
subfield = "1908"
print("Before: ", df_country_subfield.loc[country, subfield])
print("After: ", df_country_subfield_unique.loc[country, subfield])

Before:  0
After:  0


In [629]:
uf.get_subfield_info(subfield)

,subfield_id,subfield_name,field_name,domain_name
0,1908,Geophysics,Earth and Planetary Sciences,Physical Sciences


In [630]:
px.histogram(df_country_stats_unique, x="log_total_articles")

In [631]:
big_countries = df_country_stats_unique.query("total_articles > 1000").index.to_list()

In [632]:
least_collab_dependent_subfields = df_country_subfield_diff.dropna(how="all").mean(axis=0).sort_values().head(20).index.astype(int).to_list()
most_collab_dependent_subfields = df_country_subfield_diff.dropna(how="all").mean(axis=0).sort_values(ascending=False).head(20).index.astype(int).to_list()

least_collab_dependent_countries = df_country_subfield_diff.dropna(how="all").mean(axis=1).sort_values().head(20).index.to_list()
most_collab_dependent_countries = df_country_subfield_diff.dropna(how="all").mean(axis=1).sort_values(ascending=False).head(20).index.to_list()
most_collab_dependent_big_countries = df_country_subfield_diff.loc[big_countries].dropna(how="all").mean(axis=1).sort_values(ascending=False).head(20).index.to_list()

In [633]:
# (
#     uf.df_topics
#     [["subfield_id", "subfield_name", "field_id", "field_name", "domain_id", "domain_name"]]
#     .drop_duplicates()
#     .query("subfield_id == @least_collab_dependent_subfields")
# )

(
    uf.df_country
    [["alpha-2", "name", "region", "sub-region"]]
    .drop_duplicates()
    .rename(columns={"alpha-2": "country"})
    .query("country == @most_collab_dependent_big_countries")
)

,country,name,region,sub-region
14,AT,Austria,Europe,Western Europe
21,BE,Belgium,Europe,Western Europe
34,BG,Bulgaria,Europe,Eastern Europe
44,CL,Chile,Americas,Latin America and the Caribbean
59,CZ,Czechia,Europe,Eastern Europe
60,DK,Denmark,Europe,Northern Europe
75,FI,Finland,Europe,Northern Europe
100,HK,Hong Kong,Asia,Eastern Asia
101,HU,Hungary,Europe,Eastern Europe
107,IE,Ireland,Europe,Northern Europe


In [634]:
x_labels = df_country_subfield_diff.columns.to_list()
# y_labels = df_country_subfield_diff.index.to_list()
y_labels = most_collab_dependent_countries + most_collab_dependent_big_countries + least_collab_dependent_countries
uf.plotly_heatmap(
    df_country_subfield_diff.loc[y_labels, x_labels],
    x_labels=x_labels,
    y_labels=y_labels,
    x_type="subfield",
    y_type="country",
    title="Percent of articles loss by subfield (when excluding collaborative articles)"
)

# Compare Interest Metrics

In [635]:
df_interest_metric = uf.get_interest_metric(df_country_subfield)
df_interest_metric_unique = uf.get_interest_metric(df_country_subfield_unique)

In [636]:
df_interest_metric

,1100,1102,1103,1104,1105,1106,1107,1108,1109,1110,...,3603,3604,3605,3607,3608,3609,3611,3612,3614,3616
country,,,,,,,,,,,,,,,,,,,,,
AE,NaN,1.616804,0.415345,NaN,NaN,-0.309071,NaN,NaN,0.167542,-1.615803,...,NaN,NaN,NaN,NaN,NaN,NaN,1.365982,NaN,NaN,1.119661
AF,NaN,3.341553,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.207558,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AL,NaN,NaN,NaN,NaN,NaN,2.434247,NaN,NaN,2.217713,1.532981,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AM,NaN,NaN,NaN,0.856195,NaN,-0.403481,NaN,NaN,NaN,-1.017065,...,NaN,0.697934,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
XK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
YE,NaN,NaN,NaN,NaN,1.961432,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ZA,0.670424,1.198094,1.208576,0.608855,1.230585,0.096394,1.15285,1.881929,0.573007,0.311131,...,0.170551,-0.137192,0.161779,NaN,1.501157,0.718779,0.436446,NaN,-0.085808,0.660129


In [637]:
# subset_countries = uf.top_n_countries_by_articles(20)
# subset_subfields = df_interest_metric.columns
# uf.plotly_heatmap(
#     df_interest_metric.loc[subset_countries, subset_subfields],
#     x_labels=subset_subfields,
#     y_labels=subset_countries,
#     x_type="subfield",
#     y_type="country",
# )

In [638]:
# subset_countries = uf.top_n_countries_by_articles(20)
# subset_subfields = df_interest_metric_unique.columns
# uf.plotly_heatmap(
#     df_interest_metric_unique.loc[subset_countries, subset_subfields],
#     x_labels=subset_subfields,
#     y_labels=subset_countries,
#     x_type="subfield",
#     y_type="country",
# )

In [639]:
df_interest_stats = (
    df_interest_metric
    .mean(axis=1)
    .to_frame("mean_val")
    .assign(
        median_val = df_interest_metric.median(axis=1),
        std_val = df_interest_metric.std(axis=1),
        # gini = df_interest_metric.apply(uf.gini, axis=1),
    )
)

df_interest_stats_unique = (
    df_interest_metric_unique
    .mean(axis=1)
    .to_frame("mean_val")
    .assign(
        median_val = df_interest_metric_unique.median(axis=1),
        std_val = df_interest_metric_unique.std(axis=1),
        # gini = df_interest_metric_unique.apply(uf.gini, axis=1),
    )
)

df_interest_metric_compare = (
    df_interest_stats
    .merge(df_interest_stats_unique, left_index=True, right_index=True, how="outer", suffixes=("", "_unique"))
)

In [640]:
(
    df_interest_metric_compare
    .loc[big_countries]
    .dropna(subset=["mean_val_unique"])
    .assign(
        mean_diff = lambda df: df.mean_val - df.mean_val_unique,
        median_diff = lambda df: df.median_val - df.median_val_unique,
        std_diff = lambda df: df.std_val - df.std_val_unique
    )
    .dropna(subset=["std_diff"])
    .sort_values("std_diff")
)

,mean_val,median_val,std_val,mean_val_unique,median_val_unique,std_val_unique,mean_diff,median_diff,std_diff
country,,,,,,,,,
SI,-0.022991,0.032073,0.752727,-0.019132,0.087513,0.840039,-0.003860,-0.055440,-0.087313
KR,-0.404153,-0.304810,0.806443,-0.463582,-0.352458,0.883694,0.059429,0.047648,-0.077251
HR,-0.015816,-0.063120,0.890855,-0.019457,-0.123340,0.965863,0.003641,0.060220,-0.075008
AT,-0.125218,-0.079072,0.620127,-0.104441,-0.049095,0.694068,-0.020777,-0.029977,-0.073941
IT,-0.283060,-0.118110,0.693957,-0.272697,-0.093035,0.760271,-0.010363,-0.025075,-0.066314
DK,-0.103674,-0.068918,0.622432,-0.089682,-0.037437,0.686190,-0.013992,-0.031481,-0.063758
TR,-0.109938,-0.117579,0.780440,-0.116591,-0.116718,0.842120,0.006653,-0.000860,-0.061680
PL,-0.426057,-0.434250,0.849347,-0.416683,-0.375408,0.904011,-0.009374,-0.058842,-0.054664
JP,-0.510060,-0.270192,0.937676,-0.538487,-0.285397,0.990482,0.028427,0.015205,-0.052806


In [641]:
uf.get_country_info("SA")

,name,region,sub-region,code
195,Saudi Arabia,Asia,Western Asia,SA


In [642]:
# px.histogram(df_interest_metric_compare, x=["mean", "mean_unique"], barmode="group")

In [643]:
# px.histogram(df_interest_metric_compare, x=["std_val", "std_val_unique"], barmode="group")

In [644]:
(
    df_interest_metric
    .sub(df_interest_metric_unique)
    .abs()
    .div(df_interest_metric.abs())
    .fillna(0)
    # .mean(axis=1)
)

,1100,1102,1103,1104,1105,1106,1107,1108,1109,1110,...,3603,3604,3605,3607,3608,3609,3611,3612,3614,3616
country,,,,,,,,,,,,,,,,,,,,,
AE,0.000000,0.000000,1.195821,0.000000,0.000000,0.000000,0.000000,0.000000,3.189999,0.328103,...,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.261433,0.0,0.000000,0.295373
AF,0.000000,0.054494,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.176948,...,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
AG,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
AL,0.000000,0.000000,0.000000,0.000000,0.000000,0.243717,0.000000,0.000000,0.000000,0.159772,...,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
AM,0.000000,0.000000,0.000000,0.563426,0.000000,1.032003,0.000000,0.000000,0.000000,0.465574,...,0.0,0.393055,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
XK,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
YE,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
ZA,0.140858,0.004942,0.102862,0.526926,0.042938,0.860660,0.351681,0.307782,0.139366,0.086250,...,0.0,0.821803,1.807734,0.0,0.051699,0.215424,0.318255,0.0,2.528637,0.170431


In [670]:
px.bar(
    df_interest_metric
    .sub(df_interest_metric_unique)
    .abs()
    # .div(df_interest_metric.abs())
    .fillna(0)
    .mean(axis=1)
)

In [666]:
uf.get_country_info("SK")

,name,region,sub-region,code
202,Slovakia,Europe,Eastern Europe,SK


In [667]:
# country_list = ["US"]
country_list = ["HK"]
# country_list = ["CH"]
# country_list = ["CL"]
country_list = ["SK"]

In [668]:
x = df_interest_metric.columns.to_list()

fig = go.Figure()
colors = qualitative.Plotly

for country in country_list:
    color = colors[(len(fig.data) // 2) % len(colors)]

    y1 = df_interest_metric.loc[country].values
    y1_safe = [v if not np.isnan(v) else None for v in y1]  # handle NaNs

    y2 = df_interest_metric_unique.loc[country].values
    y2_safe = [v if not np.isnan(v) else None for v in y2]  # handle NaNs

    fig.add_trace(go.Scatter(
        x=x,
        y=y1_safe,
        mode="lines",
        line=dict(color=color),
        name=f"{uf.id2name_country[country]} - all articles",
        hovertext=[uf.id2subfield_topic[int(s)] for s in x],
    ))
    fig.add_trace(go.Scatter(
        x=x,
        y=y2_safe,
        mode="lines",
        name=f"{uf.id2name_country[country]} - unique articles",
        line=dict(dash="dash", color=color),
        hovertext=[uf.id2subfield_topic[int(s)] for s in x],
    ))


fig.update_layout(
    title="Interest measure with or without collaborations",
    xaxis=dict(title="Sub-field"),
    yaxis=dict(title="Sub-field share"),
    legend=dict(x=0.01, y=0.99),
    margin=dict(l=60, r=60, t=40, b=40)
)

fig.show()

In [574]:
country_list = ["US", "CN", "GB"]  # list of countries

In [575]:
x = df_interest_metric_unique.columns.to_list()

fig = go.Figure()

for country in country_list:
    y = df_interest_metric_unique.loc[country].values
    y_safe = [v if not np.isnan(v) else None for v in y]  # handle NaNs

    fig.add_trace(go.Scatter(
        x=x,
        y=y_safe,
        mode="lines",
        name=f"{uf.id2name_country[country]}"
    ))

fig.update_layout(
    title="Interest distributions for selected countries (unique articles)",
    xaxis=dict(title="Sub-field"),
    yaxis=dict(title="Sub-field share"),
    legend=dict(x=0.01, y=0.99),
    margin=dict(l=60, r=60, t=40, b=40)
)

fig.show()

In [576]:
# subset_countries = (most_collab_dependent_countries +
#                     most_collab_dependent_big_countries +
#                     least_collab_dependent_countries)
subset_countries = uf.top_n_countries_by_articles(30)

subset_subfields = df_interest_metric_unique.columns.to_list()

uf.plotly_heatmap(
    df_interest_metric_unique.loc[subset_countries, subset_subfields],
    x_labels=subset_subfields,
    y_labels=subset_countries,
    x_type="subfields",
    y_type="countries",
    z_min=-3, z_max=3, colorscale="symmetric",
    title="Interest metric"
)

# Cosine distances

In [577]:
df_interest_metric_unique

,1100,1102,1103,1104,1105,1106,1107,1108,1109,1110,...,3603,3604,3605,3607,3608,3609,3611,3612,3614,3616
country,,,,,,,,,,,,,,,,,,,,,
AE,-0.141324,0.250685,-0.458084,0.817151,NaN,-0.104415,NaN,NaN,NaN,-0.837127,...,NaN,0.647821,0.254382,NaN,NaN,NaN,1.064201,NaN,NaN,0.068773
AF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.771716,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AL,NaN,NaN,NaN,NaN,0.252742,0.263719,NaN,NaN,NaN,-0.692137,...,NaN,1.996784,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AM,NaN,NaN,NaN,NaN,0.211069,-0.183418,NaN,NaN,0.105759,NaN,...,NaN,0.163352,0.868525,NaN,NaN,NaN,NaN,NaN,NaN,0.682917
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
XK,NaN,NaN,NaN,NaN,NaN,1.187128,NaN,NaN,NaN,0.231271,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.057250,NaN
YE,NaN,NaN,1.465981,NaN,NaN,0.433356,NaN,NaN,1.415681,0.576112,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.303478,NaN
ZA,0.606150,0.694476,0.882454,0.391904,0.919764,0.182535,1.223860,NaN,0.805486,0.415442,...,1.159073,-0.182891,-0.096757,-1.088636,NaN,0.420583,0.782056,NaN,0.279589,0.074310


In [578]:
df_distance = uf.get_cosine_distances(df_interest_metric.fillna(0))
df_distance_unique = uf.get_cosine_distances(df_interest_metric_unique.fillna(0))

/Users/irinavorobeva/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/irinavorobeva/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/irinavorobeva/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul

/Users/irinavorobeva/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/irinavorobeva/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/irinavorobeva/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul



In [579]:
df_distance_diff = df_distance.sub(df_distance_unique).div(df_distance)

In [580]:
subset_countries = uf.top_n_countries_by_articles(30)
# subset_countries = df_distance_diff.index.to_list()
# subset_countries = (most_collab_dependent_countries +
#                     most_collab_dependent_big_countries +
#                     least_collab_dependent_countries)

uf.plotly_heatmap(
    df_distance_diff.loc[subset_countries, subset_countries],
    x_labels=subset_countries,
    y_labels=subset_countries,
    x_type="country",
    y_type="country",
    x_name="Country X",
    y_name="Country Y",
    title="Relative distance difference when delete collab articles (positive - closer, negative - further)",
    z_min=-0.2, z_max=0.2
)

In [581]:
subset_countries = uf.top_n_countries_by_articles(30)

uf.plotly_heatmap(
    df_distance_unique.loc[subset_countries, subset_countries],
    x_labels=subset_countries,
    y_labels=subset_countries,
    x_type="country",
    y_type="country",
    x_name="Country X",
    y_name="Country Y",
    colorscale="symmetric",
    title="Cosine distance (unique articles): 0 - same topics, 1 - orthogonal, 2 - opposite",
    z_min=0, z_max=2
)

In [582]:
# country_list = ["IN", "GB"]  # list of countries
# country_list = ["CN", "BE"]  # list of countries
country_list = ["CN", "US"]  # list of countries
# country_list = ["CN", "KR"]
# country_list = ["US", "GB"]
country_list = ["MX", "TR"]

x = df_interest_metric_unique.columns.to_list()

fig = go.Figure()

for country in country_list:
    y = df_interest_metric_unique.loc[country].values
    y_safe = [v if not np.isnan(v) else None for v in y]  # handle NaNs

    fig.add_trace(go.Scatter(
        x=x,
        y=y_safe,
        mode="lines",
        hovertext=[uf.id2subfield_topic[int(s)] for s in x],
        name=f"{uf.id2name_country[country]}"
    ))

fig.update_layout(
    title="Interest distributions for selected countries (unique articles)",
    xaxis=dict(title="Sub-field"),
    yaxis=dict(title="Sub-field share"),
    legend=dict(x=0.01, y=0.99),
    margin=dict(l=60, r=60, t=40, b=40)
)

fig.show()

# Clustering -- to be updated

In [583]:
df_dist = df_distance_unique

In [584]:
# Example: distance matrix (symmetric, zeros on diagonal)
D = df_dist.values

# Convert to condensed form (required by linkage)
condensed_D = squareform(D)

# Perform hierarchical clustering
# method can be: 'single', 'complete', 'average', 'ward'
Z = linkage(condensed_D, method='average')

In [585]:
# Assign clusters by specifying a max distance threshold

labels = fcluster(Z, t=1, criterion='distance') # -- interesting results (and threshold makes sense); bigger - whole world
# labels = fcluster(Z, t=0.9, criterion='distance') # -- smallest threshold with adequate sized clusters

order = np.argsort(labels)

In [586]:
labels = np.array(labels)

order = np.argsort(labels)
D_ordered = D[np.ix_(order, order)]

names = df_dist.index.to_numpy()
names_ordered = [uf.id2name_country[name] for name in names[order]]
labels_ordered = labels[order]

In [587]:
fig = go.Figure(
    data=go.Heatmap(
        z=D_ordered,
        x=names_ordered,
        y=names_ordered,
        colorscale="RdBu",
        colorbar=dict(title="Distance"),
        hovertemplate="Row: %{y}<br>Col: %{x}<br>Dist: %{z:.3f}<extra></extra>",
        zmin=0,
        zmax=2
    )
)

fig.update_layout(
    title="Distance matrix reordered by cluster labels",
    xaxis=dict(
        tickangle=45,
        tickfont=dict(size=8),
        automargin=True
    ),
    yaxis=dict(
        tickfont=dict(size=8),
        automargin=True
    ),
    width=900,
    height=900
)
# cluster boundaries
changes = np.where(np.diff(labels_ordered) != 0)[0] + 1

for c in changes:
    fig.add_shape(
        type="line",
        x0=-0.5, x1=len(names_ordered)-0.5,
        y0=c-0.5, y1=c-0.5,
        line=dict(color="red", width=1)
    )
    fig.add_shape(
        type="line",
        x0=c-0.5, x1=c-0.5,
        y0=-0.5, y1=len(names_ordered)-0.5,
        line=dict(color="red", width=1)
    )

fig.show()


In [588]:
hovertext = [uf.id2subfield_topic[int(x)] for x in df_country_subfield_norm.columns]
hovertext_2d = np.tile(hovertext, (len(names_ordered), 1))

fig = go.Figure(
    data=go.Heatmap(
        z=df_interest_metric_unique.fillna(0).iloc[order, :].values,
        y=names_ordered,
        x=df_country_subfield.columns,
        colorscale="RdBu",
        colorbar=dict(title="Values"),
        zmin=-2,
        zmax=2,
        text=hovertext_2d,        # 1D list for columns
        hoverinfo='x+y+text+z' # show column name, row, hovertext, and value
    )
)

fig.update_layout(
    title="Interest reordered by cluster labels",
    xaxis=dict(
        tickangle=45,
        tickfont=dict(size=8),
        automargin=True
    ),
    yaxis=dict(
        tickfont=dict(size=8),
        automargin=True
    ),
    width=900,
    height=900
)
# cluster boundaries
changes = np.where(np.diff(labels_ordered) != 0)[0] + 1

for c in changes:
    fig.add_shape(
        type="line",
        y0=c-0.5, y1=c-0.5,
        x0=-0.5, x1=len(df_country_subfield.columns)-0.5,
        line=dict(color="red", width=1)
    )

fig.show()

In [589]:
df_map = (
    df_dist
    .merge(uf.df_country[["alpha-2", "alpha-3", "name", "region", "sub-region"]], left_index=True, right_on="alpha-2", how="left")
    [["alpha-2", "alpha-3", "name", "sub-region", "region"]]
    .assign(cluster=labels,
            cluster_str = lambda df: df["cluster"].astype(str))
    .rename(columns={"alpha-3": "country", "alpha-2": "country2"})
)
df_map.sample(5)

,country2,country,name,sub-region,region,cluster,cluster_str
233.0,AE,ARE,United Arab Emirates,Western Asia,Asia,1,1
183.0,RU,RUS,Russian Federation,Eastern Europe,Europe,1,1
72.0,FK,FLK,Falkland Islands (Malvinas),Latin America and the Caribbean,Americas,3,3
197.0,RS,SRB,Serbia,Southern Europe,Europe,1,1
114.0,JO,JOR,Jordan,Western Asia,Asia,1,1


In [590]:
fig = px.choropleth(
    df_map,
    locations="country",
    color="cluster_str",               # use the categorical version
    locationmode="ISO-3",
    color_discrete_sequence=px.colors.qualitative.Set3,  # discrete color palette
    title="Country clusters based on cosine distance matrix"
)

fig.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type="natural earth"
    ),
    height=600,
)

fig.show()

In [547]:
cluster = 1

(
    df_map
    .query(f"cluster == {cluster}")
    .sort_values(["region", "sub-region"])
    .dropna()
)

,country2,country,name,sub-region,region,cluster,cluster_str
3.0,DZ,DZA,Algeria,Northern Africa,Africa,1,1
65.0,EG,EGY,Egypt,Northern Africa,Africa,1,1
150.0,MA,MAR,Morocco,Northern Africa,Africa,1,1
226.0,TN,TUN,Tunisia,Northern Africa,Africa,1,1
115.0,KZ,KAZ,Kazakhstan,Central Asia,Asia,1,1
218.0,TJ,TJK,Tajikistan,Central Asia,Asia,1,1
238.0,UZ,UZB,Uzbekistan,Central Asia,Asia,1,1
45.0,CN,CHN,China,Eastern Asia,Asia,1,1
100.0,HK,HKG,Hong Kong,Eastern Asia,Asia,1,1
112.0,JP,JPN,Japan,Eastern Asia,Asia,1,1


In [454]:
df_top_subfields, df_unique_subfields, df_subfields_counted, df_fields_counted = uf.get_cluster_dfs(
    cluster,
    df_interest_metric_unique,
    df_map
)

In [455]:
color_by_domain = {
    "Social Sciences": "red",
    "Health Sciences": "green",
    "Physical Sciences": "blue",
    "Life Sciences": "purple",
}

In [456]:
px.bar(df_fields_counted, x="field_name", y="count_subfields", color="domain_name",
       category_orders={"field_name": df_fields_counted.sort_values("count_subfields", ascending=False)["field_name"]},
       color_discrete_map=color_by_domain)